In [27]:
import pandas as pd 
import numpy as np

In [28]:
# read data 
df1 = pd.read_parquet("cleaned1.parquet")
df2 = pd.read_parquet("cleaned2.parquet")
df3 = pd.read_parquet("cleaned3.parquet")

In [29]:
df1c = df1.copy()
df2c = df2.copy()
df3c = df3.copy()

### Data Cleaning

In [30]:
df1 = df1[df1['Faulty'] == False]
df2 = df2[df2['Faulty'] == False]
df3 = df3[df3['Faulty'] == False]

In [31]:
# Data Cleanings
drop_columns_common = ["building_no", "Fan_on_group", "Cumulative_fan_on_mins", "Date", "Time", "Year", "Cumulative_damper_open_mins"]
drop_columns_df1 = drop_columns_common + ["Slab_temp", "Dew_temp", "Slab_temp_diff", "Dew_temp_diff", "Damper_open_group"]
drop_columns_df2 = drop_columns_common
drop_columns_df3 = drop_columns_common + ["Damper_open_group", "Louver_open_group", "Cumulative_louver_open_mins"]

In [32]:
# Conditional Dropping = creates a list of columns to drop
df1 = df1.drop(columns=[col for col in drop_columns_df1 if col in df1.columns])
df2 = df2.drop(columns=[col for col in drop_columns_df2 if col in df2.columns]) 
df3 = df3.drop(columns=[col for col in drop_columns_df3 if col in df3.columns])

In [33]:
# Extracting DOY from the Datetime Column; creates new column Day_of_year in each DF
df1['Day_of_Year'] = df1['Datetime'].dt.dayofyear
df2['Day_of_Year'] = df2['Datetime'].dt.dayofyear
df3['Day_of_Year'] = df3['Datetime'].dt.dayofyear

In [34]:
# creates new col Minutes_Past_midnight in all df that represents number of minutes have passed since midnight
df1['Minutes_Past_Midnight'] = df1['Datetime'].dt.hour * 60 + df1['Datetime'].dt.minute
df2['Minutes_Past_Midnight'] = df2['Datetime'].dt.hour * 60 + df2['Datetime'].dt.minute
df3['Minutes_Past_Midnight'] = df3['Datetime'].dt.hour * 60 + df3['Datetime'].dt.minute

In [35]:
for df in [df1, df2, df3]:
    ints = [i for i in range(len(df.Zone_name.unique()))]
    for i, zone in zip(ints, df.Zone_name.unique()):
        df.loc[df.Zone_name==zone,'Zone_name'] = i
    df.Zone_name = df.Zone_name.astype(int)
    df['Year'] = df.Datetime.dt.year
    
df1.Fan_status = df1.Fan_status.apply(lambda x: 0 if x=='Off' else 1 if x=='On' else np.nan).astype(float)
df2.Fan_status = df2.Fan_status.apply(lambda x: 0 if x=='Off' else 1 if x=='On' else np.nan).astype(float)
df3.Fan_status = df3.Fan_status.apply(lambda x: 0 if x=='Off' else 1 if x=='On' else np.nan).astype(float)

df1 = df1[[col for col in df1 if df1[col].dtype in [int, float, bool]]]
df2 = df2[[col for col in df2 if df2[col].dtype in [int, float, bool]]]
df3 = df3[[col for col in df3 if df3[col].dtype in [int, float, bool]]]

### Additional processing for buildings required for deep learning

In [36]:
# fill NaN values with mean
# building 1
fill_columns1 = ['Ambient_temp','Damper_status','Ambient_temp_diff']
for col in fill_columns1:
    df1[col] = df1[col].fillna(df1[col].mean())
# building 2
fill_columns2 = ["Zone_c02","Ambient_temp_diff"]
for col in fill_columns2:
    df2[col] = df2[col].fillna(df2[col].mean())
# building 3
fill_columns3 = ["Datetime_diff_mins","Zone_temp_diff","Slab_temp_diff","Dew_temp_diff","Ambient_temp_diff"]
for col in fill_columns3:
    df3[col] = df3[col].fillna(df3[col].mean())

C:\Users\tousi\AppData\Local\Temp\ipykernel_19844\3724472727.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df3[col] = df3[col].fillna(df3[col].mean())


In [37]:
df1_encoded = df1.copy()
df2_encoded = df2.copy()
df3_encoded = df3.copy()

In [38]:
# one-hot encode for categorical data
from sklearn.preprocessing import OneHotEncoder
# building 1
encoder1 = OneHotEncoder(sparse_output=False)
encoded1_array = encoder1.fit_transform(df1_encoded[['Zone_name','Season']])
encoded_df1 = pd.DataFrame(encoded1_array,columns=encoder1.get_feature_names_out(['Zone_name', 'Season']))
df1_encoded = df1_encoded.drop(columns=['Zone_name','Season'])
# building 2
encoder2 = OneHotEncoder(sparse_output=False)
encoded2_array = encoder2.fit_transform(df2_encoded[['Zone_name','Season']])
encoded_df2 = pd.DataFrame(encoded2_array,columns=encoder2.get_feature_names_out(['Zone_name', 'Season']))
df2_encoded = df2_encoded.drop(columns=['Zone_name','Season'])
# building 3
encoder3 = OneHotEncoder(sparse_output=False)
encoded3_array = encoder3.fit_transform(df3[['Zone_name','Season']])
encoded_df3 = pd.DataFrame(encoded3_array,columns=encoder3.get_feature_names_out(['Zone_name','Season']))
df3_encoded = df3_encoded.drop(columns=['Zone_name','Season'])

In [145]:
import pickle
with open("onehot_encoder_building1","wb") as file:
    pickle.dump(encoder1,file)
with open("onehot_encoder_building2","wb") as file:
    pickle.dump(encoder2,file)
with open("onehot_encoder_building3","wb") as file:
    pickle.dump(encoder3,file)

In [39]:
# merge encoded dataset
df1_encoded = pd.concat([df1_encoded,encoded_df1],axis = 1)
df2_encoded = pd.concat([df2_encoded,encoded_df2],axis = 1)
df3_encoded = pd.concat([df3_encoded,encoded_df3],axis = 1)

In [40]:
# drop faulty column
df1_encoded = df1_encoded.drop(columns=['Faulty'])
df2_encoded = df2_encoded.drop(columns=['Faulty'])
df3_encoded = df3_encoded.drop(columns=['Faulty'])

In [41]:
# minmax scale non-categorical data
from sklearn.preprocessing import MinMaxScaler
# building 1
scale1_columns = ['Zone_temp', 'Ambient_temp', 'Damper_status', 'Fan_status','Datetime_diff_mins', 'Zone_temp_diff', 'Ambient_temp_diff']
scaler1 = MinMaxScaler()
scaled1_data = scaler1.fit_transform(df1_encoded[scale1_columns])
scaled1_df = pd.DataFrame(scaled1_data,columns=scale1_columns)
# building 2
scaler2 = MinMaxScaler()
scale2_columns = ['Zone_temp', 'Slab_temp', 'Dew_temp', 'Ambient_temp', 'Zone_c02',
       'Fan_status', 'Datetime_diff_mins', 'Zone_temp_diff', 'Slab_temp_diff',
       'Dew_temp_diff', 'Ambient_temp_diff']
scaled2_data = scaler2.fit_transform(df2_encoded[scale2_columns])
scaled2_df = pd.DataFrame(scaled2_data,columns=scale2_columns)
# building 3
scaler3 = MinMaxScaler()
scale3_columns = ['Zone_temp', 'Slab_temp', 'Dew_temp', 'Ambient_temp', 'Zone_c02',
       'Damper_status', 'Fan_status', 'Datetime_diff_mins', 'Zone_temp_diff',
       'Slab_temp_diff', 'Dew_temp_diff', 'Ambient_temp_diff']
scaled3_data = scaler3.fit_transform(df3_encoded[scale3_columns])
scaled3_df = pd.DataFrame(scaled3_data,columns=scale3_columns)


In [146]:
with open("minmax_building1","wb") as file:
    pickle.dump(scaler1,file)
with open("minmax_building2","wb") as file:
    pickle.dump(scaler2,file)
with open("minmax_building3","wb") as file:
    pickle.dump(scaler3,file)

In [42]:
# drop columns which were scaled using minmax scaler
df1_encoded = df1_encoded.drop(columns=scale1_columns)
df2_encoded = df2_encoded.drop(columns=scale2_columns)
df3_encoded = df3_encoded.drop(columns=scale3_columns)

In [43]:
# merge one-hot encoded data and minmax scaled data together
df1_encoded = pd.concat([df1_encoded,scaled1_df],axis=1)
df2_encoded = pd.concat([df2_encoded,scaled2_df],axis=1)
df3_encoded = pd.concat([df3_encoded,scaled3_df],axis=1)
print(df1_encoded.columns)
print(df2_encoded.columns)
print(df3_encoded.columns)

Index(['Zone_name_0', 'Zone_name_1', 'Zone_name_2', 'Zone_name_3',
       'Zone_name_4', 'Zone_name_5', 'Zone_name_6', 'Zone_name_7',
       'Zone_name_8', 'Zone_name_9', 'Zone_name_10', 'Zone_name_11',
       'Zone_name_12', 'Zone_name_13', 'Zone_name_14', 'Zone_name_15',
       'Zone_name_16', 'Zone_name_17', 'Zone_name_18', 'Zone_name_19',
       'Zone_name_20', 'Zone_name_21', 'Season_1', 'Season_2', 'Season_3',
       'Season_4', 'Zone_temp', 'Ambient_temp', 'Damper_status', 'Fan_status',
       'Datetime_diff_mins', 'Zone_temp_diff', 'Ambient_temp_diff'],
      dtype='object')
Index(['Zone_name_0', 'Season_1', 'Season_2', 'Season_3', 'Season_4',
       'Zone_temp', 'Slab_temp', 'Dew_temp', 'Ambient_temp', 'Zone_c02',
       'Fan_status', 'Datetime_diff_mins', 'Zone_temp_diff', 'Slab_temp_diff',
       'Dew_temp_diff', 'Ambient_temp_diff'],
      dtype='object')
Index(['Zone_name_0', 'Zone_name_1', 'Zone_name_2', 'Zone_name_3',
       'Zone_name_4', 'Zone_name_5', 'Zone_name_6', '

In [44]:
# drop any rows which have null values
df1_encoded = df1_encoded.dropna(subset=df1_encoded.columns)
df2_encoded = df2_encoded.dropna(subset=df2_encoded.columns)
df3_encoded = df3_encoded.dropna(subset=df3_encoded.columns)

In [45]:
# retrieve feature columns
feature1_col = [col for col in df1_encoded.columns if col is not "Fan_status"]
feature2_col = [col for col in df2_encoded.columns if col is not "Fan_status"]
feature3_col = [col for col in df3_encoded.columns if col is not "Fan_status"]
print(len(feature1_col))
print(len(feature2_col))
print(len(feature3_col))

32
15
30


<>:1: SyntaxWarning: "is not" with a literal. Did you mean "!="?
<>:2: SyntaxWarning: "is not" with a literal. Did you mean "!="?
<>:3: SyntaxWarning: "is not" with a literal. Did you mean "!="?
<>:1: SyntaxWarning: "is not" with a literal. Did you mean "!="?
<>:2: SyntaxWarning: "is not" with a literal. Did you mean "!="?
<>:3: SyntaxWarning: "is not" with a literal. Did you mean "!="?
C:\Users\tousi\AppData\Local\Temp\ipykernel_19844\2022853643.py:1: SyntaxWarning: "is not" with a literal. Did you mean "!="?
  feature1_col = [col for col in df1_encoded.columns if col is not "Fan_status"]
C:\Users\tousi\AppData\Local\Temp\ipykernel_19844\2022853643.py:2: SyntaxWarning: "is not" with a literal. Did you mean "!="?
  feature2_col = [col for col in df2_encoded.columns if col is not "Fan_status"]
C:\Users\tousi\AppData\Local\Temp\ipykernel_19844\2022853643.py:3: SyntaxWarning: "is not" with a literal. Did you mean "!="?
  feature3_col = [col for col in df3_encoded.columns if col is not "Fa

### Create training/testing datasets for each buildings

In [46]:
# define feature and target 
# building 1
model1_X = df1_encoded[feature1_col]
model1_y = df1_encoded['Fan_status']
# building 2
model2_X = df2_encoded[feature2_col]
model2_y = df2_encoded['Fan_status']
# building 3
model3_X = df3_encoded[feature3_col]
model3_y = df3_encoded['Fan_status']

In [83]:
model2_X.head()

,Zone_name_0,Season_1,Season_2,Season_3,Season_4,Zone_temp,Slab_temp,Dew_temp,Ambient_temp,Zone_c02,Datetime_diff_mins,Zone_temp_diff,Slab_temp_diff,Dew_temp_diff,Ambient_temp_diff
2341,1.0,0.0,1.0,0.0,0.0,0.242502,0.551346,0.581882,0.267442,0.151282,0.000058,0.583488,0.674667,0.519310,0.506757
2342,1.0,0.0,1.0,0.0,0.0,0.246331,0.551346,0.577526,0.264535,0.165477,0.000058,0.584416,0.674667,0.511093,0.500000
2343,1.0,0.0,1.0,0.0,0.0,0.264837,0.550349,0.577526,0.264535,0.159141,0.000058,0.605751,0.673333,0.519310,0.506757
2344,1.0,0.0,1.0,0.0,0.0,0.255265,0.550349,0.577526,0.264535,0.150132,0.000058,0.564935,0.674667,0.519310,0.506757
2345,1.0,0.0,1.0,0.0,0.0,0.248245,0.549352,0.573171,0.261628,0.155907,0.000058,0.568646,0.673333,0.511093,0.500000


In [47]:
# convert data to tensors so that they can be processed in PyTorch library
import torch
# building 1
model1_X_tensor = torch.tensor(model1_X.values, dtype=torch.float32)
model1_y_tensor = torch.tensor(model1_y.values, dtype=torch.long)
# building 2
model2_X_tensor = torch.tensor(model2_X.values, dtype=torch.float32)
model2_y_tensor = torch.tensor(model2_y.values, dtype=torch.long)
# building 3
model3_X_tensor = torch.tensor(model3_X.values, dtype=torch.float32)
model3_y_tensor = torch.tensor(model3_y.values, dtype=torch.long)

In [71]:
print(model1_X_tensor.size())
print(model2_X_tensor.size())
print(model3_X_tensor.size())

torch.Size([376821, 32])
torch.Size([22716, 15])
torch.Size([467517, 30])


In [48]:
# split dataset into training and test sets
from sklearn.model_selection import train_test_split
# building 1
model1_X_train, model1_X_test, model1_y_train, model1_y_test = train_test_split(model1_X_tensor,model1_y_tensor,test_size=0.3)
# building 2
model2_X_train, model2_X_test, model2_y_train, model2_y_test = train_test_split(model2_X_tensor,model2_y_tensor,test_size=0.3)
# building 3
model3_X_train, model3_X_test, model3_y_train, model3_y_test = train_test_split(model3_X_tensor,model3_y_tensor,test_size=0.3)

In [49]:
# Define customized data loading class
from torch.utils.data import Dataset
class CustomDataset(Dataset):
    def __init__(self,features,labels):
        self.features = features
        self.labels = labels 
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [50]:
# create datasets 
# building 1
model1_train_dataset = CustomDataset(model1_X_train,model1_y_train)
model1_test_dataset = CustomDataset(model1_X_test,model1_y_test)
# building 2
model2_train_dataset = CustomDataset(model2_X_train,model2_y_train)
model2_test_dataset = CustomDataset(model2_X_test,model2_y_test)
# building 1
model3_train_dataset = CustomDataset(model3_X_train,model3_y_train)
model3_test_dataset = CustomDataset(model3_X_test,model3_y_test)

In [72]:
# create dataloader object for each building
from torch.utils.data import DataLoader
# building 1
train1_dataloader = DataLoader(model1_train_dataset,batch_size=128,shuffle=True)
test1_dataloader = DataLoader(model1_train_dataset,batch_size=128,shuffle=True)
# building 2
train2_dataloader = DataLoader(model2_train_dataset,batch_size=32,shuffle=True)
test2_dataloader = DataLoader(model2_train_dataset,batch_size=32,shuffle=True)
# building 3
train3_dataloader = DataLoader(model3_train_dataset,batch_size=128,shuffle=True)
test3_dataloader = DataLoader(model3_train_dataset,batch_size=128,shuffle=True)

### Build deep learning model for each building

In [63]:
# define model for each building
from torch import nn
# feed-forward network for building 1
class building1_FFN(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden1 = nn.Linear(32,20)
        self.hidden2 = nn.Linear(20,12)
        self.hidden3 = nn.Linear(12,6)
        self.output = nn.Linear(6,2)
    def forward(self,x):
        x = torch.relu(self.hidden1(x))
        x = torch.relu(self.hidden2(x))
        x = torch.relu(self.hidden3(x))
        x = self.output(x)
        return x
    
# feed-forward network for building 2
class building2_FFN(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden1 = nn.Linear(15,10)
        self.hidden2 = nn.Linear(10,5)
        self.output = nn.Linear(5,2)
    def forward(self,x):
        x = torch.relu(self.hidden1(x))
        x = torch.relu(self.hidden2(x))
        x = self.output(x)
        return x
    
# feed-forward network for building 3
class building3_FFN(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden1 = nn.Linear(30,18)
        self.hidden2 = nn.Linear(18,10)
        self.hidden3 = nn.Linear(10,6)
        self.output = nn.Linear(6,2)
    def forward(self,x):
        x = torch.relu(self.hidden1(x))
        x = torch.relu(self.hidden2(x))
        x = torch.relu(self.hidden3(x))
        x = self.output(x)
        return x

In [131]:
# define train loop function
def train_loop(model, dataloader, optimiser, loss_fn):
    model.train()
    for batch, (X,y) in enumerate(dataloader):
        pred = model(X)
        loss = loss_fn(pred,y)
        loss.backward()
        optimiser.step()
        optimiser.zero_grad()

        if batch % 100 == 0:
            print(f"Batch {batch}: {loss}")

# define test loop function
def test_loop(model,dataloader,loss_fn):
    model.eval()
    num_batches = len(dataloader)
    with torch.no_grad():
        total = 0
        correct = 0
        loss = 0
        for X, y in dataloader:
            pred = model(X)
            loss += loss_fn(pred,y).item()
            predicted_classes = pred.argmax(dim=1)
            correct += (predicted_classes == y).sum().item()
            total += y.size(0)
    accuracy = (correct/total) * 100
    loss /= num_batches
    
    print(f"Accuracy: {round(accuracy,3)}, loss: {round(loss,3)}")
    acc = round(accuracy,3)
    return acc


In [133]:
# building 1 model parameters
model1 = building1_FFN()
learning_rate = 0.01 
epoch = 10
optimizer = torch.optim.Adam(model1.parameters(),lr=learning_rate)
loss = torch.nn.CrossEntropyLoss()

In [134]:
accuracy1 = []
# loop to train and test model for building 1
for i in range(epoch):
    ep = i + 1
    print(f"Epoch {i+1}")
    train_loop(model1,train1_dataloader,optimizer,loss)
    acc = test_loop(model1,test1_dataloader,loss)
    accuracy1.append([ep,acc])
print("Done")

Epoch 1
Batch 0: 0.7054619193077087
Batch 100: 0.41581225395202637
Batch 200: 0.3861415386199951
Batch 300: 0.3623194098472595
Batch 400: 0.3506670594215393
Batch 500: 0.38340267539024353
Batch 600: 0.35393497347831726
Batch 700: 0.3225581645965576
Batch 800: 0.3666840195655823
Batch 900: 0.2736600935459137
Batch 1000: 0.3944739103317261
Batch 1100: 0.3924742341041565
Batch 1200: 0.35114797949790955
Batch 1300: 0.3738558292388916
Batch 1400: 0.31600505113601685
Batch 1500: 0.3396950960159302
Batch 1600: 0.3077908158302307
Batch 1700: 0.38134172558784485
Batch 1800: 0.35874125361442566
Batch 1900: 0.31889742612838745
Batch 2000: 0.3297349214553833
Accuracy: 81.613, loss: 0.333
Epoch 2
Batch 0: 0.2427450716495514
Batch 100: 0.3319282531738281
Batch 200: 0.3080708086490631
Batch 300: 0.36935192346572876
Batch 400: 0.33093661069869995
Batch 500: 0.36731138825416565
Batch 600: 0.31738781929016113
Batch 700: 0.2886049449443817
Batch 800: 0.2713475823402405
Batch 900: 0.31788378953933716
Batc

In [135]:
print(accuracy1)

[[1, 81.613], [2, 81.542], [3, 82.992], [4, 84.004], [5, 83.638], [6, 84.086], [7, 84.074], [8, 85.32], [9, 83.575], [10, 86.213]]


In [136]:
# building 2 model parameters
model2 = building2_FFN()
learning_rate = 0.01 
epoch = 10
optimizer = torch.optim.Adam(model2.parameters(),lr=learning_rate)
loss = torch.nn.CrossEntropyLoss()

In [137]:
accuracy2 = []
# loop to train and test model for building 2
for i in range(epoch):
    ep = i + 1
    print(f"Epoch {i+1}")
    train_loop(model2,train2_dataloader,optimizer,loss)
    acc = test_loop(model2,test2_dataloader,loss)
    accuracy2.append([ep,acc])
print("Done")

Epoch 1
Batch 0: 0.7851142883300781
Batch 100: 0.05306478217244148
Batch 200: 0.13140122592449188
Batch 300: 0.34256136417388916
Batch 400: 0.19224998354911804
Accuracy: 95.013, loss: 0.183
Epoch 2
Batch 0: 0.05278920754790306
Batch 100: 0.08244523406028748
Batch 200: 0.12896862626075745
Batch 300: 0.17024432122707367
Batch 400: 0.06949392706155777
Accuracy: 95.013, loss: 0.174
Epoch 3
Batch 0: 0.10651522129774094
Batch 100: 0.2192724347114563
Batch 200: 0.20693480968475342
Batch 300: 0.1439731866121292
Batch 400: 0.14213396608829498
Accuracy: 95.013, loss: 0.168
Epoch 4
Batch 0: 0.11758881062269211
Batch 100: 0.13652828335762024
Batch 200: 0.1582821011543274
Batch 300: 0.13555952906608582
Batch 400: 0.06451371312141418
Accuracy: 95.013, loss: 0.169
Epoch 5
Batch 0: 0.230331689119339
Batch 100: 0.1465412974357605
Batch 200: 0.1942174881696701
Batch 300: 0.18396174907684326
Batch 400: 0.1478564739227295
Accuracy: 95.013, loss: 0.163
Epoch 6
Batch 0: 0.1458507925271988
Batch 100: 0.20596

In [138]:
# building 3 model parameters
model3 = building3_FFN()
learning_rate = 0.01 
epoch = 10
optimizer = torch.optim.Adam(model3.parameters(),lr=learning_rate)
loss = torch.nn.CrossEntropyLoss()

In [139]:
accuracy3 = []
# loop to train and test model for building 3
for i in range(epoch):
    ep = i + 1
    print(f"Epoch {i+1}")
    train_loop(model3,train3_dataloader,optimizer,loss)
    acc = test_loop(model3,test3_dataloader,loss)
    accuracy3.append([ep,acc])
print("Done")

Epoch 1
Batch 0: 0.6363947987556458
Batch 100: 0.04665806144475937
Batch 200: 0.012241947464644909
Batch 300: 0.04761403426527977
Batch 400: 0.0021499625872820616
Batch 500: 0.038045357912778854
Batch 600: 0.009145949967205524
Batch 700: 0.021943481639027596
Batch 800: 0.026929261162877083
Batch 900: 0.011622997932136059
Batch 1000: 0.006512003019452095
Batch 1100: 0.009400768205523491
Batch 1200: 0.0034667891450226307
Batch 1300: 0.014623446390032768
Batch 1400: 0.024652546271681786
Batch 1500: 0.050302036106586456
Batch 1600: 0.0014137066900730133
Batch 1700: 0.004568641539663076
Batch 1800: 0.0641721859574318
Batch 1900: 0.005793213844299316
Batch 2000: 0.03877990320324898
Batch 2100: 0.00962365884333849
Batch 2200: 0.014783741906285286
Batch 2300: 0.022551575675606728
Batch 2400: 0.007246989291161299
Batch 2500: 0.12592878937721252
Accuracy: 99.209, loss: 0.031
Epoch 2
Batch 0: 0.06024063006043434
Batch 100: 0.04098162055015564
Batch 200: 0.03381318226456642
Batch 300: 0.0040679196

In [140]:
torch.save(model1.state_dict(),"model1_parameters.pth")
torch.save(model2.state_dict(),"model2_parameters.pth")
torch.save(model3.state_dict(),"model3_parameters.pth")

In [141]:
building1_analysis = pd.DataFrame(accuracy1,columns=['epoch','accuracy'])
building2_analysis = pd.DataFrame(accuracy2,columns=['epoch','accuracy'])
building3_analysis = pd.DataFrame(accuracy3,columns=['epoch','accuracy'])

In [144]:
building3_analysis

,epoch,accuracy
0,1,99.209
1,2,99.338
2,3,99.388
3,4,99.386
4,5,99.405
5,6,99.416
6,7,99.432
7,8,99.393
8,9,99.435
9,10,99.441


In [128]:
loaded_model = building2_FFN()
loaded_model.state_dict(torch.load("model2_parameters.pth"))
loaded_model.eval()
input_data = [1,0,1,0,0,0.3,0.5,0.5,0.3,0.1,0.5,0.09,0.5,0.7,0.9]
input_tensor = torch.FloatTensor(input_data).unsqueeze(0)

with torch.no_grad():
    output = loaded_model(input_tensor)

prediction = output
print(prediction)
print(prediction.argmax(dim=1).item())

tensor([[ 0.0305, -0.0713]])
0


C:\Users\tousi\AppData\Local\Temp\ipykernel_19844\1441325408.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  loaded_model.state_dict(torch.load("model2_parameters.pth"))